# 第8讲：优化思想、支持向量机与超参数（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：7分钟

        ## 目标

        1. 修改COMPARISON_PARAMS，比较较小和较大的C、gamma组合及其验证AP与边界；
2. 检查最佳组合附近的验证表现是否稳定，不只抄写best_params_；
3. 解释为什么StandardScaler必须放进Pipeline、测试集不能参与搜索；

        学生版的代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。教师版和参考答案版已保存一次完整运行结果。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.sans-serif": ["Microsoft YaHei", "Hiragino Sans GB", "Arial Unicode MS", "DejaVu Sans"],
    "axes.unicode_minus": False,
})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 建立时间边界与完整Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix
traffic = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"]).sort_values(["timestamp", "detector_id"])
data = traffic.reset_index(drop=True)
features = ["flow_15min", "occupancy_pct"]
target = "severe_congestion"

# 按完整时间戳切分：同一时刻的六个检测器不会分到训练和测试两侧。
unique_times = np.sort(data["timestamp"].unique())
cut = int(len(unique_times) * .75)
split_time = unique_times[cut]
train = data[data["timestamp"] < split_time].copy().reset_index(drop=True)
test = data[data["timestamp"] >= split_time].copy().reset_index(drop=True)

def make_time_block_splits(frame, n_splits=4):
    # 在唯一时间戳上前推切分，再映射回各检测器的数据行。
    times = np.sort(frame["timestamp"].unique())
    splits = []
    for train_time_idx, valid_time_idx in TimeSeriesSplit(n_splits=n_splits).split(times):
        train_times = times[train_time_idx]
        valid_times = times[valid_time_idx]
        train_idx = np.flatnonzero(frame["timestamp"].isin(train_times).to_numpy())
        valid_idx = np.flatnonzero(frame["timestamp"].isin(valid_times).to_numpy())
        assert frame.iloc[train_idx]["timestamp"].max() < frame.iloc[valid_idx]["timestamp"].min()
        splits.append((train_idx, valid_idx))
    return splits

CV_SPLITS = make_time_block_splits(train, n_splits=4)
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC(kernel="rbf", class_weight="balanced")),
])
print("训练时间", train["timestamp"].min(), "至", train["timestamp"].max())
print("测试时间", test["timestamp"].min(), "至", test["timestamp"].max())
print("训练/测试严重拥堵比例", round(train[target].mean(), 3), round(test[target].mean(), 3))
print("训练期检测器记录数", train["detector_id"].value_counts().sort_index().to_dict())
for fold, (train_idx, valid_idx) in enumerate(CV_SPLITS, 1):
    print(f"第{fold}折：", train.iloc[train_idx]["timestamp"].max(), "<", train.iloc[valid_idx]["timestamp"].min())

## 3. 用时间序列交叉验证联合搜索C与gamma

In [ ]:
C_GRID = [0.1, 0.5, 2, 8, 32]          # TODO：可缩小为三个值后比较计算预算
GAMMA_GRID = [0.01, 0.05, 0.2, 1, 5]  # TODO：比较gamma过小与过大
PARAM_GRID = {"svc__C": C_GRID, "svc__gamma": GAMMA_GRID}
search = GridSearchCV(
    pipe, PARAM_GRID,
    cv=CV_SPLITS,
    scoring="average_precision",
    n_jobs=1,
    return_train_score=True,
)
search.fit(train[features], train[target])
results = (pd.DataFrame(search.cv_results_)[
    ["param_svc__C", "param_svc__gamma", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]
    ].sort_values("rank_test_score"))
print("最佳组合", search.best_params_, "验证折平均AP", round(search.best_score_, 3))
print("训练期正类比例（随机排序AP基线）", round(train[target].mean(), 3))
display(results.head(10).round(3))
results.to_csv(OUTPUT_DIR / "lesson08_cv_results.csv", index=False)

## 4. 把验证结果画成热图

In [ ]:
heat = (results.pivot(index="param_svc__gamma", columns="param_svc__C", values="mean_test_score")
        .reindex(index=GAMMA_GRID, columns=C_GRID))
fig, ax = plt.subplots(figsize=(8.2, 4.8))
im = ax.imshow(heat.to_numpy(), cmap="YlGnBu", aspect="auto")
for row in range(heat.shape[0]):
    for col in range(heat.shape[1]):
        ax.text(col, row, f"{heat.iloc[row, col]:.3f}", ha="center", va="center",
                color="white" if heat.iloc[row, col] > heat.to_numpy().mean() else "#202124")
ax.set_xticks(range(len(C_GRID)), C_GRID); ax.set_yticks(range(len(GAMMA_GRID)), GAMMA_GRID)
ax.set(xlabel="C", ylabel="gamma", title="按完整时间块前推验证：平均精确率（AP）")
fig.colorbar(im, ax=ax, label="验证折平均AP")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson08_cv_heatmap.png", dpi=180); plt.show()

## 5. 比较指定参数组合的验证AP与边界

In [ ]:
# TODO：修改其中一组C和gamma，再比较图中的边界与验证AP。
COMPARISON_PARAMS = [
    (0.1, 0.2),   # 较小C
    (32, 0.2),    # 较大C
    (2, 0.01),    # 较小gamma
    (2, 5),       # 较大gamma
]

x1 = np.linspace(train[features[0]].quantile(.01), train[features[0]].quantile(.99), 120)
x2 = np.linspace(train[features[1]].quantile(.01), train[features[1]].quantile(.99), 120)
xx, yy = np.meshgrid(x1, x2)
grid = pd.DataFrame({features[0]: xx.ravel(), features[1]: yy.ravel()})
display_sample = pd.concat([
    train[train[target] == 0].sample(450, random_state=RANDOM_STATE),
    train[train[target] == 1],
]).sample(frac=1, random_state=RANDOM_STATE)

fig, axes = plt.subplots(2, 2, figsize=(10, 7.6), sharex=True, sharey=True)
comparison_rows = []
for ax, (c_value, gamma_value) in zip(axes.ravel(), COMPARISON_PARAMS):
    row = results[(results["param_svc__C"] == c_value) & (results["param_svc__gamma"] == gamma_value)]
    if row.empty:
        raise ValueError("COMPARISON_PARAMS中的取值必须先出现在C_GRID和GAMMA_GRID中")
    cv_ap = float(row.iloc[0]["mean_test_score"])
    model = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(kernel="rbf", C=c_value, gamma=gamma_value, class_weight="balanced")),
    ]).fit(train[features], train[target])
    zz = model.decision_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz > 0, levels=[-.5, .5, 1.5], alpha=.18, cmap="coolwarm")
    ax.contour(xx, yy, zz, levels=[0], colors=["#C30D23"], linewidths=2)
    ax.scatter(display_sample[features[0]], display_sample[features[1]],
               c=display_sample[target], s=9, cmap="coolwarm", alpha=.45)
    ax.set_title(f"C={c_value}, gamma={gamma_value}, 验证AP={cv_ap:.3f}")
    comparison_rows.append({"C": c_value, "gamma": gamma_value, "validation_AP": cv_ap})
for ax in axes[-1, :]: ax.set_xlabel(features[0])
for ax in axes[:, 0]: ax.set_ylabel(features[1])
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson08_parameter_boundaries.png", dpi=180); plt.show()
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(3))
comparison_table.to_csv(OUTPUT_DIR / "lesson08_parameter_comparison.csv", index=False)

## 6. 冻结方案后评价独立测试期

In [ ]:
risk = search.best_estimator_.decision_function(test[features])
pred = (risk >= 0).astype(int)
ap = average_precision_score(test[target], risk)
auc = roc_auc_score(test[target], risk)
cm = pd.DataFrame(confusion_matrix(test[target], pred),
                  index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])
metrics = pd.DataFrame([{
    "best_cv_average_precision": search.best_score_,
    "test_average_precision": ap,
    "test_roc_auc": auc,
    "test_prevalence": test[target].mean(),
}])
display(metrics.round(3)); display(cm)
metrics.to_csv(OUTPUT_DIR / "lesson08_test_metrics.csv", index=False)
cm.to_csv(OUTPUT_DIR / "lesson08_test_confusion.csv")

## 7. 由模型实际分数绘制决策区域并自检

In [ ]:
x1 = np.linspace(data[features[0]].quantile(.01), data[features[0]].quantile(.99), 120)
x2 = np.linspace(data[features[1]].quantile(.01), data[features[1]].quantile(.99), 120)
xx, yy = np.meshgrid(x1, x2)
grid = pd.DataFrame({features[0]: xx.ravel(), features[1]: yy.ravel()})
zz = search.best_estimator_.decision_function(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz > 0, levels=[-.5, .5, 1.5], alpha=.18, cmap="coolwarm")
plt.contour(xx, yy, zz, levels=[0], colors=["#C30D23"], linewidths=2)
sample_plot = test.sample(min(500,len(test)), random_state=RANDOM_STATE)
plt.scatter(sample_plot[features[0]], sample_plot[features[1]], c=sample_plot[target], s=10, cmap="coolwarm", alpha=.55)
plt.xlabel(features[0]); plt.ylabel(features[1]); plt.title("冻结方案后的测试期SVM决策区域")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson08_svm_boundary.png", dpi=180); plt.show()
fold_time_order = all(train.iloc[a]["timestamp"].max() < train.iloc[b]["timestamp"].min() for a, b in CV_SPLITS)
detector_complete = set(train["detector_id"]) == set(traffic["detector_id"])
checks = {
    "训练早于测试": train["timestamp"].max() < test["timestamp"].min(),
    "每个验证折严格晚于训练折": fold_time_order,
    "六个检测器全部保留": detector_complete and train["detector_id"].nunique() == 6,
    "标准化位于Pipeline": list(search.best_estimator_.named_steps) == ["scale", "svc"],
    "测试期仅在冻结方案后评价": 0 <= ap <= 1,
    "七个输出文件已生成": all((OUTPUT_DIR / name).exists() for name in [
        "lesson08_cv_results.csv", "lesson08_cv_heatmap.png", "lesson08_test_metrics.csv",
        "lesson08_test_confusion.csv", "lesson08_svm_boundary.png",
        "lesson08_parameter_boundaries.png", "lesson08_parameter_comparison.csv"]),
}
status = "PASS" if all(checks.values()) else "CHECK"; (OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8"); print(status, checks)

## Checks｜当堂记录

        - 修改COMPARISON_PARAMS，比较较小和较大的C、gamma组合及其验证AP与边界
- 检查最佳组合附近的验证表现是否稳定，不只抄写best_params_
- 解释为什么StandardScaler必须放进Pipeline、测试集不能参与搜索

        **预期结果：** 交叉验证结果表与热图、四组参数边界比较、独立测试指标与混淆矩阵。

        **完成标准：** 六个检测器全部保留；验证按完整时间块前推；测试期只在方案冻结后评价；Notebook生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。